# California Housing Choropleth

This notebook demonstrates how to create a choropleth map from Census housing data using GeoPandas.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

## Load California Counties

We use the US Census TIGER counties shapefile (2022) clipped to California.

In [ ]:
# California counties from Census TIGER (FIPS state code 06)
url = 'https://www2.census.gov/geo/tiger/TIGER2022/COUNTY/tl_2022_us_county.zip'
counties = gpd.read_file(url)
ca = counties[counties['STATEFP'] == '06'].copy()
ca = ca.to_crs('EPSG:3310')  # California Albers
print(ca.shape)
ca[['NAME', 'geometry']].head()

## Simulate Housing Data

In a real workflow you would join Census ACS data here.  
For this demo we generate random median home values per county.

In [ ]:
import numpy as np
np.random.seed(42)

ca = ca.copy()
ca['median_home_value'] = np.random.randint(200_000, 1_200_000, size=len(ca))
ca[['NAME', 'median_home_value']].sort_values('median_home_value', ascending=False).head(10)

## Choropleth Map

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 10))

ca.plot(
    column='median_home_value',
    cmap='YlOrRd',
    linewidth=0.5,
    edgecolor='gray',
    legend=True,
    legend_kwds={
        'label': 'Median Home Value (USD)',
        'orientation': 'horizontal',
        'shrink': 0.6
    },
    ax=ax
)

ax.set_title('California — Median Home Value by County', fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Label Counties

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 11))

ca.plot(
    column='median_home_value',
    cmap='Blues',
    linewidth=0.5,
    edgecolor='white',
    legend=True,
    ax=ax
)

for _, row in ca.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        row['NAME'],
        xy=(centroid.x, centroid.y),
        fontsize=4,
        ha='center',
        color='black'
    )

ax.set_title('California Counties — Labeled', fontsize=13)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
ca[['NAME', 'median_home_value']].describe()